# Kanji Detector — Geração de Dataset + Treino YOLOv26n no Kaggle

**Checklist antes de rodar:**
1. Adicione o dataset **Manga109** (dataset cru original) no painel lateral direito do notebook em **+ Add Input**.
2. Habilite a GPU: *Session options → Accelerator → GPU T4 x2 ou P100*.
3. Clique em **Run All**.

In [ ]:
# Verificar GPU disponivel
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'GPU nao encontrada!')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponivel: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Instalar dependencias

In [ ]:
!pip install -q ultralytics albumentations fonttools Pillow opencv-python-headless pyyaml requests
print('Dependencias instaladas com sucesso.')

## 2. Configurar repositorio

In [ ]:
import os, sys

REPO_URL = 'https://github.com/MiguelMussalam/Detector-de-kanjis-n1.git'
WORK_DIR = '/kaggle/working'
REPO_DIR = os.path.join(WORK_DIR, 'Detector-de-kanjis-n1')

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repositorio ja existe em {REPO_DIR}')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'Diretorio de trabalho: {os.getcwd()}')

## 3. Gerar Dataset Sintético no Kaggle

O script lerá as imagens originais do Manga109 sob `/kaggle/input/`, baixará as fontes TTF necessárias automaticamente e gerará 5.000 páginas contendo kanjis aplicados aleatoriamente com as caixas delimitadoras de treino YOLO.

In [ ]:
!python -m src.kanji_detector.generate_pages

## 4. Verificar arquivos gerados

In [ ]:
import os
from config import TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_LBL_DIR, VAL_LBL_DIR

train_imgs = os.listdir(TRAIN_IMG_DIR) if os.path.exists(TRAIN_IMG_DIR) else []
val_imgs   = os.listdir(VAL_IMG_DIR) if os.path.exists(VAL_IMG_DIR) else []
train_lbls = os.listdir(TRAIN_LBL_DIR) if os.path.exists(TRAIN_LBL_DIR) else []
val_lbls   = os.listdir(VAL_LBL_DIR) if os.path.exists(VAL_LBL_DIR) else []

print('Dataset sintético gerado em /kaggle/working/data/dataset:')
print(f'  Treino:    {len(train_imgs)} imagens | {len(train_lbls)} labels')
print(f'  Validacao: {len(val_imgs)} imagens | {len(val_lbls)} labels')

## 5. Treinar a YOLOv26n

In [ ]:
!python -m src.kanji_detector.train

## 6. Mostrar resultados e curvas de aprendizado

In [ ]:
from IPython.display import Image, display
import os

runs_dir = '/kaggle/working/Detector-de-kanjis-n1/kanji_detector/run1'
print(f'Procurando resultados em: {runs_dir}')

if os.path.exists(runs_dir):
    # Curvas de treino
    results_plot = os.path.join(runs_dir, 'results.png')
    if os.path.exists(results_plot):
        print('=== Curvas de Treino (mAP / Loss) ===')
        display(Image(results_plot))

    # Matriz de confusao
    conf_matrix = os.path.join(runs_dir, 'confusion_matrix.png')
    if os.path.exists(conf_matrix):
        print('=== Matriz de Confusao ===')
        display(Image(conf_matrix))

    # Amostras preditas
    val_batch = os.path.join(runs_dir, 'val_batch0_pred.jpg')
    if os.path.exists(val_batch):
        print('=== Predicoes no conjunto de validacao ===')
        display(Image(val_batch))
else:
    print('Diretorio de resultados nao encontrado.')

## 7. Exportar modelo treinado (`best.pt`)

In [ ]:
import shutil

runs_dir   = '/kaggle/working/Detector-de-kanjis-n1/kanji_detector/run1'
best_pt    = os.path.join(runs_dir, 'weights', 'best.pt')
output_dir = '/kaggle/working/output'

if os.path.exists(best_pt):
    os.makedirs(output_dir, exist_ok=True)
    shutil.copy2(best_pt, os.path.join(output_dir, 'best.pt'))
    print(f'Modelo best.pt exportado com sucesso para: {output_dir}/best.pt')
else:
    print('Arquivo best.pt nao encontrado. O treino terminou sem salvar?')